# Notebook 5 — Orpheus-Hindi (SachinTelecmi/Orpheus-tts-hi)

- Model: https://huggingface.co/SachinTelecmi/Orpheus-tts-hi
- Base: Llama-3 + SNAC codec. Fine-tuned on Hindi + English **with code-mixed support**.
- 4-bit quantization (bitsandbytes nf4) is mandatory on T4. Budget ~20 s/sentence.

> **DO NOT reconstruct `generate_speech` from memory.**
> Open the model card on HF and copy the function VERBATIM.
> The SNAC token unpacking is fiddly and the special-token IDs are version-sensitive.


In [ ]:
# === Cell 1: install ===
!pip install -q transformers torch torchaudio bitsandbytes accelerate
!pip install -q snac soundfile


In [ ]:
# Mount Drive (or skip if running locally) and clone the audit folder.
# Adjust this cell to point AUDIT_DIR at wherever audit/ lives in your runtime.
import os
from pathlib import Path

# Two common patterns:
#   1. Colab + Drive: AUDIT_DIR = "/content/drive/MyDrive/hienglish/audit"
#   2. Colab + git clone:
#         !git clone https://github.com/<you>/hienglish.git /content/hienglish
#         AUDIT_DIR = "/content/hienglish/audit"
#   3. Local: AUDIT_DIR = str(Path.cwd().parent / "audit")  (if launched from notebooks/)

AUDIT_DIR = os.environ.get("AUDIT_DIR", "/content/audit")
assert Path(AUDIT_DIR).is_dir(), f"AUDIT_DIR={AUDIT_DIR} missing — set it before running."
print(f"AUDIT_DIR = {AUDIT_DIR}")


In [ ]:
import csv
from pathlib import Path

EVAL_TSV = Path(AUDIT_DIR) / "eval_sentences.tsv"
with open(EVAL_TSV, encoding="utf-8") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

assert len(rows) == 30, f"expected 30 sentences, got {len(rows)}"
print(f"Loaded {len(rows)} sentences from {EVAL_TSV}")
print(rows[0])


In [ ]:
# === Cell 4: load model + SNAC decoder in 4-bit ===
from pathlib import Path
import time, json
import torch
import soundfile as sf
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from snac import SNAC

MODEL_NAME = "orpheus_hi"
OUT = Path(AUDIT_DIR) / "results" / MODEL_NAME
OUT.mkdir(parents=True, exist_ok=True)

quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
REPO = "SachinTelecmi/Orpheus-tts-hi"
model = AutoModelForCausalLM.from_pretrained(
    REPO, quantization_config=quant_cfg, device_map="auto", trust_remote_code=True,
)
tok = AutoTokenizer.from_pretrained(REPO, trust_remote_code=True)
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").eval().cuda()

# Special-token IDs from the model card (verify against current card before running):
END_OF_SPEECH_TOKEN     = 128258
START_OF_HUMAN_TOKEN    = 128259
END_OF_HUMAN_TOKEN      = 128260
START_OF_AI_TOKEN       = 128261
END_OF_AI_TOKEN         = 128262
AUDIO_CODE_BASE_OFFSET  = 128266


In [ ]:
# === Cell 5: PASTE generate_speech VERBATIM from the HF model card ===
# https://huggingface.co/SachinTelecmi/Orpheus-tts-hi
#
# The function constructs the Llama prompt with the special tokens above,
# samples until END_OF_SPEECH_TOKEN, regroups the SNAC codes (3 codebooks),
# and decodes via snac_model.decode -> 24 kHz waveform.
#
# DO NOT REWRITE THIS FROM MEMORY. Copy the canonical version. Stub below
# only documents the contract — replace it before running:

def generate_speech(text: str, temperature: float = 0.4) -> "np.ndarray":
    """Returns a numpy float32 mono waveform at 24000 Hz."""
    raise NotImplementedError(
        "Replace this stub with the verbatim generate_speech() from the model card "
        "at https://huggingface.co/SachinTelecmi/Orpheus-tts-hi"
    )


In [ ]:
# === Cell 6: run inference ===
log = []
for r in rows:
    rid, cat, text = r["id"], r["category"], r["text"]
    t0 = time.time()
    try:
        audio = generate_speech(text, temperature=0.4)
        out_path = OUT / f"{rid}.wav"
        sf.write(out_path, audio, 24000)
        log.append({
            "id": rid, "category": cat,
            "elapsed_s": time.time() - t0,
            "duration_s": float(len(audio) / 24000),
            "status": "ok",
        })
    except Exception as e:
        log.append({"id": rid, "category": cat, "status": "error", "error": str(e)})
        print(f"  [error] {rid}: {e}")


In [ ]:
import json
out_log = Path(OUT) / "log.json"
with open(out_log, "w", encoding="utf-8") as f:
    json.dump(log, f, ensure_ascii=False, indent=2)

n_ok = sum(1 for x in log if x["status"] == "ok")
print(f"{MODEL_NAME}: {n_ok}/30 succeeded — log at {out_log}")
